# Gold: Dimensions Bootstrap

Bootstrap notebook (run once).

Adjusts historical SCD2 effective_from values to ensure alignment with event timestamps (cdc_ts) for the initial backfill. This guarantees valid dimension joins during the first fact load.

Do not use in regular pipeline execution.

A baseline date of 2010-01-01 is assigned to effective_from, as it is earlier than all timestamps in the dataset. This ensures consistent SCD2 joins without inferring actual entity creation times.

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
cfg = load_config()
JOB_NAMES = cfg["spark_jobs"]["jobs"]

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_dims_bootstrap"])
        .getOrCreate()
)

In [4]:
tables = [
    "polaris.gold.dim_customers_scd2",
    "polaris.gold.dim_sellers_scd2",
    "polaris.gold.dim_products_scd2",
]

baseline_ts = F.lit("2010-01-01").cast("timestamp")

for table in tables:
    # Load dimension table (Iceberg)
    dim = spark.read.format("iceberg").load(table)
    
    # Only backfill if effective_from is later than baseline
    dim_backfilled = dim.withColumn(
        "effective_from",
        F.when(F.col("effective_from") > baseline_ts, baseline_ts)
         .otherwise(F.col("effective_from")))
    
    # Overwrite table (one-time operation)
    (
        dim_backfilled.write
        .format("iceberg")
        .mode("overwrite")
        .save(table)  
    )

In [5]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 